# **ResNet50 — Baseline Final (Sintesis EXP03 + ResNet18 + ViT exp01)**

Notebook ini menyatukan praktik terbaik dari 3 sumber:
- **EXP03_Layer.ipynb** (ResNet50 lama) — starting point, tapi `OneCycleLR`-nya terbukti jadi root cause instabilitas fold (lihat diskusi sebelumnya: Fold 4 selalu menang karena satu-satunya yang tidak pernah early-stop sebelum siklus LR selesai)
- **cnn-resnet18.ipynb** — sudah pakai `timm`, Kaggle Secrets, `ReduceLROnPlateau`, struktur modular `train_one_fold()`
- **exp01-vit-base-16-amp.ipynb** — sudah pakai AMP, class weight ternormalisasi, pin_memory, cudnn.benchmark

**Yang diselaraskan lintas SEMUA arsitektur** (biar perbandingan CNN vs ViT nanti adil, tidak ada confound): FC head sederhana (`drop_rate`, bukan custom multi-layer), scheduler `ReduceLROnPlateau` dengan setting yang sama persis, kriteria checkpoint & early stopping berbasis `val_f1`, augmentasi "medium" yang identik.

## 1. Import & Setup

In [1]:
# 1. Install & Import
import os, copy, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# PERBAIKAN (dari ViT exp01): AMP -- sebagian besar operasi forward jalan di
# float16 (lebih cepat & hemat VRAM di GPU RTX/Tensor Core), backward/update tetap
# presisi lewat GradScaler. Belum ada di kedua notebook ResNet sebelumnya.
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms, datasets
from PIL import Image
from tqdm import tqdm

import timm   # PERBAIKAN: torchvision.models -> timm, biar 1 API dipakai semua arsitektur (ResNet18/50, EfficientNet, ViT, dst)
import wandb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Ambil API key dari environment variable
wandb_api_key = os.getenv("WANDB_API_KEY")

os.environ["TORCH_HOME"] = "D:/cache/torch"
os.environ["HF_HOME"] = "D:/cache/huggingface"

os.environ["WANDB_DIR"] = "D:/cache/wandb"
os.environ["WANDB_CACHE_DIR"] = "D:/cache/wandb_cache"

os.environ["TEMP"] = "D:/cache/temp"
os.environ["TMP"] = "D:/cache/temp"

os.environ["CUDA_CACHE_PATH"] = "D:/cache/cuda"

print(os.getcwd())

# Login ke wandb
wandb.login(key=wandb_api_key)

# PERBAIKAN (dari ViT exp01 + ResNet18): seed eksplisit -- EXP01-03 ResNet50 lama
# cuma nge-seed StratifiedKFold, TIDAK nge-seed init bobot FC head / urutan shuffle.
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# PERBAIKAN (dari ViT exp01): cudnn.benchmark auto-tune algoritma konvolusi
# tercepat untuk ukuran input yang konsisten (semua di-resize ke 224x224).
torch.backends.cudnn.benchmark = True


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\UNIDA\_netrc.


d:\Devianest_SkripsiTest\Code_CNN


wandb: Currently logged in as: devianestnarendra to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


## 2. Config

In [2]:
TRAIN_DIR = r"D:\Devianest_SkripsiTest\train"
TEST_DIR  = r"D:\Devianest_SkripsiTest\test"

# PERBAIKAN (VSCode lokal): ganti "/kaggle/working" -> folder "outputs" relatif
# terhadap lokasi notebook ini. os.makedirs(..., exist_ok=True) otomatis bikin
# foldernya kalau belum ada, supaya tidak error "No such file or directory"
# saat pertama kali disimpan.
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ARCH_KEY  = "resnet50"
TIMM_NAME = "resnet50"

IMG_SIZE     = 224
BATCH_SIZE   = 32          # tetap seperti EXP01-03 (ResNet50 lebih berat dari ResNet18, batch lebih kecil)
EPOCHS       = 1
N_FOLDS      = 5
DROPOUT      = 0.3         # dipertahankan dari EXP01-03 (nilai yang sudah teruji utk ResNet50)
LR           = 1e-4        # PERBAIKAN: dipertahankan nilai ResNet (BUKAN LR ViT 3e-5) -- CNN historically
                            # lebih toleran ke LR sedikit lebih tinggi dibanding attention layer ViT yang sensitif
WEIGHT_DECAY = 1e-4         # dipertahankan nilai konvensi CNN transfer learning (bukan WD ViT 0.01)
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 7     # PERBAIKAN: naik dari 5 -> 7 (lebih toleran sebelum berhenti)

WANDB_PROJECT = "SkinDisease-CNN"   # disamakan dengan notebook ResNet18, biar 1 project W&B

# ResNet50: unfreeze layer2/3/4 + fc -- sama scope kapasitas dengan EXP03 (untuk
# tetap bisa dibandingkan), TAPI training regime-nya sudah diperbaiki (lihat bawah).
UNFREEZE_PATTERNS = ["layer2", "layer3", "layer4", "fc"]

# PERBAIKAN (dari ViT exp01 + ResNet18): scheduler ReduceLROnPlateau disamakan
# PERSIS dengan setting yang sudah dipakai di ViT & ResNet18 -- root cause
# instabilitas ResNet50 lama adalah OneCycleLR yang di-set untuk siklus 50 epoch
# penuh, tapi EarlyStopping hampir selalu memotong training di epoch 11-22
# (SEBELUM fase anneal selesai) -- lihat diskusi sebelumnya soal Fold 4 yang
# selalu menang karena satu-satunya fold yang tidak pernah early-stop.
SCHEDULER_FACTOR    = 0.1
SCHEDULER_PATIENCE  = 2
SCHEDULER_THRESHOLD = 1e-4
SCHEDULER_MIN_LR    = 1e-7


## 3. Dataset & Augmentasi

In [3]:
# PERBAIKAN: augmentasi disamakan PERSIS dengan versi terbaru ViT exp01 (medium,
# termasuk RandomResizedCrop) -- bukan versi ResNet18 yang belum pakai
# RandomResizedCrop. Ini penting supaya CNN dan ViT dibandingkan dengan
# preprocessing yang identik (bukan confound tambahan).
def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf


classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

filepaths, labels = [], []
for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", num_classes)


class SkinDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


Total Images : 15557
Classes      : 23


## 4. Early Stopping (val_f1) + K-Fold

In [4]:
class EarlyStopping:
    # Kriteria val_f1 (bukan val_loss) -- konsisten dengan ViT exp01 & ResNet18:
    # lebih robust untuk data imbalanced (23 kelas DermNet) dibanding val_loss.
    def __init__(self, patience=5):
        self.patience = patience
        self.best_f1 = -np.inf
        self.counter = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


## 5. Model Builder + Freeze Strategy

In [5]:
def build_model(num_classes, dropout=DROPOUT):
    # PERBAIKAN: head sederhana bawaan timm (Linear + drop_rate), BUKAN custom
    # Linear->BN->ReLU->Dropout->Linear seperti EXP01-03 lama. Diselaraskan dengan
    # ViT exp01 & ResNet18 -- supaya kapasitas head TIDAK jadi confound tambahan
    # saat membandingkan arsitektur (kalau satu arsitektur dikasih head lebih besar
    # dari yang lain, selisih performa bisa jadi cuma soal head, bukan backbone).
    model = timm.create_model(
        TIMM_NAME, pretrained=True, num_classes=num_classes, drop_rate=dropout
    )
    return model


def apply_freeze_strategy(model, patterns=UNFREEZE_PATTERNS):
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(pat in name for pat in patterns):
            p.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  [{ARCH_KEY}] Trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")
    return model


## 6. Train 1 Fold

In [6]:
def train_one_fold(fold, train_idx, val_idx, run):
    train_tf, eval_tf = get_transforms(IMG_SIZE)

    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i] for i in val_idx]

    # PERBAIKAN (dari ViT exp01): pin_memory=True -- percepat transfer CPU->GPU.
    # num_workers=2 dari ResNet18 (lebih cepat load data daripada 0 di EXP03 lama).
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, train_tf),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
    )

    # PERBAIKAN: reseed per fold -- inisialisasi FC head & urutan shuffle
    # reproducible, tidak tergantung urutan eksekusi fold sebelumnya.
    random.seed(SEED + fold); np.random.seed(SEED + fold)
    torch.manual_seed(SEED + fold); torch.cuda.manual_seed_all(SEED + fold)

    model = build_model(num_classes)
    model = apply_freeze_strategy(model)
    model = model.to(device)

    # PERBAIKAN (dari ViT exp01): class_weights dinormalisasi supaya rata-rata = 1.
    # EXP01-03 & ResNet18 sebelumnya cuma 1./bincount TANPA normalisasi --
    # magnitude weight antar kelas timpang, jadi salah satu sumber loss yang
    # "melompat" tergantung komposisi kelas tiap batch.
    class_counts  = np.bincount(train_labels, minlength=num_classes)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * num_classes

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device), label_smoothing=LABEL_SMOOTHING
    )
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    # PERBAIKAN: scheduler disamakan persis dengan ViT exp01 & ResNet18 -- root
    # cause instabilitas ResNet50 lama (OneCycleLR + EarlyStopping yang memotong
    # sebelum siklus selesai) sudah tidak ada lagi di sini.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE,
        threshold=SCHEDULER_THRESHOLD, min_lr=SCHEDULER_MIN_LR
    )

    # PERBAIKAN (dari ViT exp01): AMP -- autocast di forward pass, GradScaler
    # untuk backward/update supaya gradient float16 tidak underflow.
    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)
    best_val_f1 = -np.inf
    best_val_loss = np.inf
    best_train_loss = np.inf
    best_model_path = None
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        print(f"\n[{ARCH_KEY} | fold {fold+1}] Epoch {epoch+1}/{EPOCHS} (LR: {optimizer.param_groups[0]['lr']:.2e})")

        # TRAIN
        model.train()
        train_loss = 0
        for imgs, tgts in tqdm(train_loader, desc="Train"):
            imgs, tgts = imgs.to(device), tgts.to(device)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), tgts)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for imgs, tgts in tqdm(val_loader, desc="Val"):
                imgs, tgts = imgs.to(device), tgts.to(device)
                with autocast():
                    outputs = model(imgs)
                    v_loss  = criterion(outputs, tgts)
                val_loss += v_loss.item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(tgts.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average="weighted", zero_division=0)
        recall    = recall_score(trues, preds, average="weighted", zero_division=0)
        f1        = f1_score(trues, preds, average="weighted", zero_division=0)

        scheduler.step(f1)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        run.log({
            "epoch": epoch + 1,
            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss": avg_val_loss,
            f"fold_{fold+1}/accuracy": acc,
            f"fold_{fold+1}/precision": precision,
            f"fold_{fold+1}/recall": recall,
            f"fold_{fold+1}/f1_score": f1,
            f"fold_{fold+1}/lr": optimizer.param_groups[0]["lr"],
        })

        # SAVE BEST MODEL -- kriteria val_f1 tertinggi (bukan val_loss terendah)
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_loss = avg_val_loss
            best_train_loss = avg_train_loss

            save_path = f"{OUTPUT_DIR}/{ARCH_KEY}_fold{fold+1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss": avg_val_loss,
                "f1": f1,
                "fold": fold + 1,
                "arch": ARCH_KEY,
            }, save_path)
            best_model_path = save_path
            print(f"  ✓ Model saved → {save_path} (F1: {f1:.4f})")

        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label="Train Loss", marker="o", markersize=3)
    ax.plot(epochs_ran, val_losses, label="Val Loss", marker="o", markersize=3)
    ax.set_title(f"{ARCH_KEY} — Fold {fold+1} Loss Curve")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    curve_path = f"{OUTPUT_DIR}/{ARCH_KEY}_Fold_{fold+1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    run.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)

    return {
        "arch": ARCH_KEY,
        "fold": fold + 1,
        "train_loss": best_train_loss,
        "val_loss": best_val_loss,
        "f1": best_val_f1,
        "model_path": best_model_path,
    }


## 7. MAIN LOOP — 5 Fold (ResNet50)

Kalau waktu habis di tengah jalan: checkpoint tiap fold udah ke-save duluan (di `all_results`), aman buat dilanjut manual per-fold.

In [7]:
all_results = []

run = wandb.init(
    project="SkinDisease-CNN",
    entity="devianestnarendra_Team",
    name=f"{ARCH_KEY}",
    reinit=True,
    config={
        "architecture": ARCH_KEY,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "optimizer": "AdamW",
        "scheduler": f"ReduceLROnPlateau(mode=max, factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})",
        "lr": LR,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "unfreeze_patterns": UNFREEZE_PATTERNS,
        "checkpoint_criteria": "best_val_f1",
        "amp": True,
        "seed": SEED,
    }
)

for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):
    result = train_one_fold(fold, train_idx, val_idx, run)
    all_results.append(result)

run.finish()

results_df = pd.DataFrame(all_results)
results_df


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

d:\Devianest_SkripsiTest\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\UNIDA\.cache\huggingface\hub\models--timm--resnet50.a1_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\UNIDA\AppData\Local\Temp\ipykernel_9992\3009854539.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...

  [resnet50] Trainable params: 23,329,815 / 23,555,159 (99.0%)

[resnet50 | fold 1] Epoch 1/1 (LR: 1.00e-04)


...

  [resnet50] Trainable params: 23,329,815 / 23,555,159 (99.0%)

[resnet50 | fold 5] Epoch 1/1 (LR: 1.00e-04)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_9992\3009854539.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_9992\3009854539.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:26<00:00,  3.70it/s]


Train Loss : 3.1914 | Val Loss  : 3.1506
Accuracy   : 0.1977  | Precision : 0.2283
Recall     : 0.1977  | F1 Score  : 0.1645
  ✓ Model saved → outputs/resnet50_fold5.pth (F1: 0.1645)


epoch,▁▁▁▁▁
fold_1/accuracy,▁
fold_1/f1_score,▁
fold_1/lr,▁
fold_1/precision,▁
fold_1/recall,▁
fold_1/train_loss,▁
fold_1/val_loss,▁
fold_2/accuracy,▁
fold_2/f1_score,▁
+26,...


,arch,fold,train_loss,val_loss,f1,model_path
0,resnet50,1,3.194749,3.148378,0.164997,outputs/resnet50_fold1.pth
1,resnet50,2,3.192308,3.166275,0.174805,outputs/resnet50_fold2.pth
2,resnet50,3,3.189819,3.181044,0.135059,outputs/resnet50_fold3.pth
3,resnet50,4,3.190327,3.153477,0.162559,outputs/resnet50_fold4.pth
4,resnet50,5,3.191381,3.150619,0.164483,outputs/resnet50_fold5.pth


## 8. Rekap 5-Fold

In [8]:
print(f"Arsitektur : {ARCH_KEY}")
print(f"Mean F1    : {results_df['f1'].mean():.4f}")
print(f"Std F1     : {results_df['f1'].std():.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(results_df["fold"].astype(str), results_df["f1"], color="#1f3a5f")
ax.axhline(results_df["f1"].mean(), color="red", linestyle="--", label=f"Mean = {results_df['f1'].mean():.3f}")
ax.set_xlabel("Fold"); ax.set_ylabel("Val F1-Score")
ax.set_title(f"{ARCH_KEY} — F1-Score per Fold")
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_fold_f1_chart.png", dpi=200)
plt.show()

results_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_all_folds.csv", index=False)


Arsitektur : resnet50
Mean F1    : 0.1604
Std F1     : 0.0149


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_9992\3901584725.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Test Evaluation (Fold Terbaik)

In [9]:
best_fold_result = max(all_results, key=lambda r: r["f1"])
best_overall_path = best_fold_result["model_path"]
print(f"Fold terbaik    : {best_fold_result['fold']}")
print(f"Checkpoint      : {best_overall_path}")
print(f"Val F1 terbaik  : {best_fold_result['f1']:.4f}")

_, eval_tf = get_transforms(IMG_SIZE)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = build_model(num_classes)
model = model.to(device)
checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc="Test"):
        imgs = imgs.to(device)
        with autocast():
            out = model(imgs)
        y_true.extend(tgts.numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(cmap="Blues", ax=ax, xticks_rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Confusion_Matrix.png", dpi=150, bbox_inches="tight")
plt.show()

test_summary_df = pd.DataFrame([{
    "arch": ARCH_KEY, "Accuracy": acc, "Precision": precision, "Recall": recall, "F1": f1
}])
test_summary_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv", index=False)
print(f"\n✓ Test summary disimpan -> {OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv")


Fold terbaik    : 2
Checkpoint      : outputs/resnet50_fold2.pth
Val F1 terbaik  : 0.1748


Test:   0%|          | 0/126 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_9992\3000596476.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Test: 100%|██████████| 126/126 [00:47<00:00,  2.67it/s]


Accuracy  : 0.1944
Precision : 0.2676
Recall    : 0.1944
F1-Score  : 0.1760

                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.57      0.21      0.30       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.39      0.16      0.23       288
                                          Atopic Dermatitis Photos       0.20      0.08      0.12       123
                                            Bullous Disease Photos       0.00      0.00      0.00       113
                Cellulitis Impetigo and other Bacterial Infections       0.10      0.05      0.07        73
                                                     Eczema Photos       0.27      0.23      0.25       309
                                      Exanthems and Drug Eruptions       0.12      0.43      0.19       101
                 Hair Loss Photos Alopecia and other Hair 

C:\Users\UNIDA\AppData\Local\Temp\ipykernel_9992\3000596476.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Catatan

- Checkpoint format: `{model_state_dict, val_loss, f1, fold, arch}` -- kompatibel dengan notebook ResNet18 dan rencana perbandingan 5 arsitektur.
- `results_df` punya kolom yang sama (`arch, fold, train_loss, val_loss, f1, model_path`) -- tinggal `pd.concat()` dengan hasil arsitektur lain saat rekap akhir.
- **Rekomendasi lanjutan**: notebook ResNet18 kamu saat ini augmentasinya belum pakai `RandomResizedCrop` seperti di sini -- worth diselaraskan juga supaya semua arsitektur (termasuk ViT) benar-benar dilatih dengan preprocessing identik.
- Scope unfreeze (`layer2/3/4 + fc`) dipertahankan sama dengan EXP03 lama supaya masih bisa dibandingkan sebagai referensi "apakah masalahnya training regime atau kapasitas", sesuai hipotesis yang kita uji sebelumnya.